# Thuisbatterij-simulatie V2
Verschillen met V1: de batterij ontlaadt alleen in uren zonder teruglevering (PV levert
niets over), en het batterijvermogen is instelbaar (standaard 2,5 kW).
Uitvoer: `batterij_v2_per_dag.csv` met dagwaarden voor de Excel-analyse.

In [ ]:
import sqlite3, os, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# ---- INSTELLINGEN ----
ACCU_KW = 2.5        # laad- en ontlaadvermogen batterij (kW)
RENDEMENT = 0.90     # round-trip rendement
DREMPEL = 0.99       # 'ideaal' = kleinste capaciteit die 99% van het haalbare levert
DAGEN = 400          # hoeveel historie inlezen
GEEN_TERUG = 0.01    # kWh per uur: daaronder telt het uur als 'PV uit'
# ----------------------

KANDIDATEN = ["/config/home-assistant_v2.db", "/homeassistant/home-assistant_v2.db",
              "/homeassistant_config/home-assistant_v2.db"]
DB = next((p for p in KANDIDATEN if os.path.exists(p)), None)
assert DB, "Database niet gevonden"
IMPORT_IDS = ["sensor.slimmemeter_energy_delivered_tariff_1", "sensor.slimmemeter_energy_delivered_tariff_2"]
EXPORT_IDS = ["sensor.slimmemeter_energy_returned_tariff_1", "sensor.slimmemeter_energy_returned_tariff_2"]
print("Database:", DB, "| accuvermogen:", ACCU_KW, "kW")

In [ ]:
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

def reeks(sid):
    mid, unit = con.execute("SELECT id, unit_of_measurement FROM statistics_meta WHERE statistic_id=?", (sid,)).fetchone()
    since = time.time() - DAGEN * 86400
    df = pd.read_sql("SELECT start_ts, sum FROM statistics WHERE metadata_id=? AND start_ts>=? ORDER BY start_ts",
                     con, params=(mid, since))
    idx = pd.DatetimeIndex(pd.to_datetime(df.start_ts, unit="s", utc=True)).tz_convert("Europe/Amsterdam")
    s = pd.Series(df["sum"].values, index=idx).diff().clip(lower=0)
    return s / 1000 if unit == "Wh" else s

def som(ids):
    out = None
    for sid in ids:
        out = reeks(sid) if out is None else out.add(reeks(sid), fill_value=0)
    return out

uur = pd.DataFrame({"afname": som(IMPORT_IDS), "teruglevering": som(EXPORT_IDS)}).dropna()
uur = uur.asfreq("h", fill_value=0)
uur["pv_uit"] = uur.teruglevering < GEEN_TERUG
uur["afname_pv_uit"] = uur.afname.where(uur.pv_uit, 0)
print(f"{uur.index.min():%d-%m-%Y} t/m {uur.index.max():%d-%m-%Y}, {len(uur)} uren")

## Simulatie
Laden uit teruglevering (max ACCU_KW per uur), ontladen alleen in uren zonder teruglevering
(max ACCU_KW per uur). Venster per dag: eerste terugleveruur van dag d tot het eerste
terugleveruur van dag d+1; de batterij start dan leeg.

In [ ]:
eff = RENDEMENT ** 0.5

def geleverd(imp, exp, uit, cap):
    soc = tot = 0.0
    for i, e, u in zip(imp, exp, uit):
        if cap > soc:
            laad = min(e, ACCU_KW, (cap - soc) / eff)
            soc += laad * eff
        if u:
            ont = min(i, ACCU_KW, soc * eff)
            soc -= ont / eff
            tot += ont
    return tot

def ideaal(imp, exp, uit):
    vol = geleverd(imp, exp, uit, 1e6)
    if vol < 0.05:
        return 0.0
    lo, hi = 0.0, 60.0
    for _ in range(25):
        mid = (lo + hi) / 2
        lo, hi = (lo, mid) if geleverd(imp, exp, uit, mid) >= DREMPEL * vol else (mid, hi)
    return hi

dagen = sorted(set(uur.index.normalize()))
start = {}
for d in dagen:
    dag = uur.loc[d:d + pd.Timedelta(hours=23)]
    t = dag.index[~dag.pv_uit]
    start[d] = t[0] if len(t) else None

rows = []
for k, d in enumerate(dagen[:-1]):
    kal = uur.loc[d:d + pd.Timedelta(hours=23)]
    r = {"datum": d.tz_localize(None).date(),
         "afname_kwh": kal.afname.sum(), "teruglevering_kwh": kal.teruglevering.sum(),
         "afname_pv_uit_kwh": kal.afname_pv_uit.sum(),
         "venster_teruglevering_kwh": 0.0, "venster_uren_teruglevering": 0,
         "venster_afname_pv_uit_kwh": 0.0, "venster_uren_pv_uit": 0,
         "ideaal_sim_kwh": 0.0}
    s = start[d]
    if s is not None:
        e = start[dagen[k + 1]] or dagen[k + 1] + pd.Timedelta(hours=12)
        w = uur.loc[s:e - pd.Timedelta(hours=1)]
        r.update(venster_teruglevering_kwh=w.teruglevering.sum(),
                 venster_uren_teruglevering=int((~w.pv_uit).sum()),
                 venster_afname_pv_uit_kwh=w.afname_pv_uit.sum(),
                 venster_uren_pv_uit=int(w.pv_uit.sum()),
                 ideaal_sim_kwh=ideaal(w.afname.values, w.teruglevering.values, w.pv_uit.values))
    rows.append(r)

per_dag = pd.DataFrame(rows).set_index("datum")
per_dag.index = pd.DatetimeIndex(per_dag.index)
seizoen = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Lente", 4: "Lente", 5: "Lente",
           6: "Zomer", 7: "Zomer", 8: "Zomer", 9: "Herfst", 10: "Herfst", 11: "Herfst"}
per_dag["seizoen"] = per_dag.index.month.map(seizoen)
per_dag.round(3).to_csv("batterij_v2_per_dag.csv", sep=";", decimal=",")
per_dag.describe(percentiles=[.5, .8, .9]).round(1)

## Verdeling over het jaar en per seizoen

In [ ]:
x = per_dag.ideaal_sim_kwh
bins = np.arange(0, max(2, np.ceil(x.max())) + 1, 1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(x, bins=bins, color="#2a7ab0", edgecolor="white")
for p, ls in [(50, "--"), (80, ":")]:
    v = np.percentile(x, p); ax.axvline(v, color="k", ls=ls, label=f"P{p} = {v:.1f} kWh")
ax.set_xlabel("ideale accugrootte (kWh)"); ax.set_ylabel("aantal dagen")
ax.set_title(f"V2, {ACCU_KW} kW, {len(per_dag)} dagen"); ax.legend(); plt.show()

fig, axs = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, s in zip(axs.flat, ["Winter", "Lente", "Zomer", "Herfst"]):
    xs = per_dag.loc[per_dag.seizoen == s, "ideaal_sim_kwh"]
    ax.hist(xs, bins=bins, color="#e08a2c", edgecolor="white")
    if len(xs): ax.axvline(xs.median(), color="k", ls="--")
    ax.set_title(f"{s}: {len(xs)} dagen, mediaan {xs.median():.1f} kWh" if len(xs) else s)
plt.tight_layout(); plt.show()
per_dag.groupby("seizoen").ideaal_sim_kwh.describe(percentiles=[.5, .8]).round(1)

## Jaaropbrengst per batterijgrootte (doorlopend)

In [ ]:
caps = [0, 2, 4, 5, 6, 8, 10, 12, 15, 20]
schaal = 365 / max(len(per_dag), 1)
opbr = pd.DataFrame([(c, geleverd(uur.afname.values, uur.teruglevering.values, uur.pv_uit.values, c) * schaal)
                     for c in caps], columns=["kWh_capaciteit", "verschoven_kWh_per_jaar"])
opbr["extra_per_kWh_capaciteit"] = opbr.verschoven_kWh_per_jaar.diff() / opbr.kWh_capaciteit.diff()
opbr.to_csv("batterij_v2_opbrengst.csv", sep=";", decimal=",", index=False)
opbr.round(0)